# Mamba + Workspace POC — Kaggle T4 x2

Run cells B and D (the go/no-go pair) **in parallel** on two T4 GPUs.

**Setup before running:**
1. Settings → Accelerator → **GPU T4 x2** (must select dual GPU)
2. Settings → Internet → **On** (needed for pip install + git clone)
3. Add a Kaggle Secret called `github_token` with a GitHub Personal Access Token
   that has `repo` scope (Settings → Secrets → Add Secret)

The notebook clones your private GitHub repo, uses the pure-PyTorch Mamba2
fallback (no `mamba-ssm` needed), and enables fp16 AMP for T4.

Cell B runs on GPU 0, Cell D runs on GPU 1 — both train simultaneously,
halving total wall time from ~24h to ~12h.

In [ ]:
# Install dependencies — pure-PyTorch path, no mamba-ssm needed
!pip install -q einops pyyaml wandb numpy

In [ ]:
import os
import subprocess
import shutil

# Clone the private repo using a GitHub token stored in Kaggle Secrets
# Before running: add a Kaggle Secret named "github_token" with a GitHub PAT (repo scope)
# Settings → Secrets → Add Secret → Name: github_token, Value: <your PAT>

repo_root = '/kaggle/working/jasper'
work_dir = os.path.join(repo_root, 'mamba-poc')
repo_url = 'https://github.com/rbf22/jasper.git'

# Clean up stale state from previous runs (e.g. moved folders)
stale = ['/kaggle/working/mamba-poc']
for s in stale:
    if os.path.exists(s):
        print(f"Removing stale dir: {s}")
        shutil.rmtree(s)

if os.path.exists(os.path.join(repo_root, '.git')):
    # Already cloned — pull latest and restore any missing files
    print(f"Repo exists, pulling latest...")
    subprocess.run(['git', '-C', repo_root, 'reset', '--hard', 'HEAD'], check=True)
    subprocess.run(['git', '-C', repo_root, 'pull'], check=True)
else:
    # Remove partial clone if any
    if os.path.exists(repo_root):
        shutil.rmtree(repo_root)
    # Clone with token auth from Kaggle Secrets
    try:
        import kaggle_secrets
        user_secrets = kaggle_secrets.UserSecretsClient()
        gh_token = user_secrets.get_secret("github_token")
        authed_url = repo_url.replace('https://', f'https://{gh_token}@')
        print(f"Cloning private repo with token auth...")
        subprocess.run(['git', 'clone', authed_url, repo_root], check=True)
    except Exception as e:
        print(f"Kaggle Secrets auth failed: {e}")
        print("Falling back to public clone (will fail if repo is private)...")
        subprocess.run(['git', 'clone', repo_url, repo_root], check=True)

os.chdir(work_dir)
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir('.'))

In [ ]:
# Verify GPU and environment
import torch
n_gpus = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPUs visible: {n_gpus}")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
print(f"Torch: {torch.__version__}")

assert n_gpus >= 2, f"Need 2 GPUs for parallel training, found {n_gpus}. Set Accelerator → GPU T4 x2."

# T4 is Turing — no bf16, but fp16 works great with AMP
# train.py auto-detects CUDA and enables fp16 autocast + GradScaler
# Cell B will run on GPU 0, Cell D on GPU 1 (via CUDA_VISIBLE_DEVICES)

In [ ]:
# Quick data test — verifies all 3 task generators
!python data.py

In [ ]:
# Param count check — verify all 4 cells are ~30M
!python model.py

In [ ]:
# Smoke test — 5M params, Task 1 depth-4, ~200 steps (~5 min on T4)
# Confirms loss drops and model generates verifiable answers
!python train.py --config configs/cell_d_kaggle.yaml --smoke-test

## Train Cell B + Cell D in Parallel (GPU 0 + GPU 1)

Each cell runs on its own T4 via `CUDA_VISIBLE_DEVICES`. Output is streamed to log files so you can monitor progress. Both save checkpoints to `checkpoints/` with different filenames (`cellB_latest.pt`, `cellD_latest.pt`) — no conflict.

~12 hours total wall time instead of ~24h sequential.

In [ ]:
import subprocess
import os
import glob
import time

# Remove stale smoke-test checkpoints (d_model=128) before real training
for ckpt in glob.glob('checkpoints/cell*_latest.pt'):
    print(f"Removing stale checkpoint: {ckpt}")
    os.remove(ckpt)

# Launch Cell B on GPU 0 and Cell D on GPU 1 simultaneously
# PYTHONUNBUFFERED=1 ensures logs flush immediately
env_b = os.environ.copy()
env_b['CUDA_VISIBLE_DEVICES'] = '0'
env_b['PYTHONUNBUFFERED'] = '1'
env_d = os.environ.copy()
env_d['CUDA_VISIBLE_DEVICES'] = '1'
env_d['PYTHONUNBUFFERED'] = '1'

log_b = open('train_cellB.log', 'w', buffering=1)
log_d = open('train_cellD.log', 'w', buffering=1)

print("Launching Cell B on GPU 0 and Cell D on GPU 1...")
proc_b = subprocess.Popen(
    ['python', '-u', 'train.py', '--config', 'configs/cell_b_kaggle.yaml'],
    env=env_b, stdout=log_b, stderr=subprocess.STDOUT
)
proc_d = subprocess.Popen(
    ['python', '-u', 'train.py', '--config', 'configs/cell_d_kaggle.yaml'],
    env=env_d, stdout=log_d, stderr=subprocess.STDOUT
)

print(f"Cell B PID: {proc_b.pid} (GPU 0)")
print(f"Cell D PID: {proc_d.pid} (GPU 1)")
print("Training in background. This cell stays alive with periodic updates.\n")

# Keep cell alive with periodic status until both finish
while proc_b.poll() is None or proc_d.poll() is None:
    time.sleep(300)  # 5 minutes
    b_status = "RUNNING" if proc_b.poll() is None else f"DONE (exit {proc_b.returncode})"
    d_status = "RUNNING" if proc_d.poll() is None else f"DONE (exit {proc_d.returncode})"
    print(f"[{time.strftime('%H:%M:%S')}] Cell B: {b_status} | Cell D: {d_status}")
    # Show last 3 lines of each log
    for name, logfile in [("B", "train_cellB.log"), ("D", "train_cellD.log")]:
        if os.path.exists(logfile):
            with open(logfile) as f:
                lines = f.readlines()
            if lines:
                for line in lines[-3:]:
                    print(f"  [{name}] {line.rstrip()}")

log_b.close()
log_d.close()

print(f"\nCell B exit code: {proc_b.returncode}")
print(f"Cell D exit code: {proc_d.returncode}")
if proc_b.returncode != 0:
    print("Cell B FAILED — check train_cellB.log")
if proc_d.returncode != 0:
    print("Cell D FAILED — check train_cellD.log")
if proc_b.returncode == 0 and proc_d.returncode == 0:
    print("Both training runs completed successfully!")

## Monitor Training Progress

Run this cell anytime to check if training is still going and see live logs.

In [ ]:
# Show training logs and process status
import os
import subprocess

# Check if processes are still running
result = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
train_procs = [l for l in result.stdout.split('\n') if 'train.py' in l and 'grep' not in l]
print(f"Running train.py processes: {len(train_procs)}")
for p in train_procs:
    print(f"  {p[:120]}")

for name, logfile in [("Cell B (GPU 0)", "train_cellB.log"), ("Cell D (GPU 1)", "train_cellD.log")]:
    print(f"\n=== {name} ===")
    if os.path.exists(logfile):
        with open(logfile) as f:
            lines = f.readlines()
        if lines:
            print(f"  ({len(lines)} lines total, showing last 30)")
            for line in lines[-30:]:
                print(line, end='')
        else:
            print("  Log file is empty — process may still be starting up")
    else:
        print("  Log not found")

## Analysis (R2, R3, R4)

Run after the training cell above finishes. Uses Cell D's checkpoint.

In [ ]:
## Analysis (R2, R3, R4)

Run after the training cell above finishes. Uses Cell D's checkpoint.

In [ ]:
# Run all analyses on Cell D
# R2: K sweep (accuracy vs inference K at each depth)
# R3: Linear probes (decodability of workspace slots)
# R4: Selective ablation (J-space signature)
!python probe.py --checkpoint checkpoints/cellD_latest.pt --config configs/cell_d_kaggle.yaml --all

In [ ]:
# Save checkpoints to Kaggle output — survives session end
# After the session ends, you can download these from the Output tab
import shutil
import os

# Copy checkpoints
if os.path.exists('checkpoints'):
    for f in os.listdir('checkpoints'):
        src = os.path.join('checkpoints', f)
        dst = os.path.join('/kaggle/working', f)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"Saved: {dst} ({os.path.getsize(src) / 1e6:.1f} MB)")

# Also save any probe output files
for fname in os.listdir('.'):
    if fname.startswith('probe_') and (fname.endswith('.json') or fname.endswith('.csv') or fname.endswith('.png')):
        shutil.copy2(fname, os.path.join('/kaggle/working', fname))
        print(f"Saved: {fname}")

print("\nAll outputs saved to /kaggle/working/ — download from the Output tab after session ends.")